# Copyright (C) 2026 Bruno Proença de Souza
# Licenciado sob GNU AGPL v3 — veja o arquivo LICENSE

# Clustering nao supervisionado com K-Means em dados climaticos brutos

Este notebook e uma copia da versao academica, ajustada para rodar com os dados brutos. O objetivo e identificar grupos com padroes climaticos decendiais semelhantes usando exclusivamente K-Means, sem alterar a matriz climatica de entrada.

**Protocolo metodologico**

- Unidade de analise: linha bruta da base `dataset_final.parquet`.
- Variaveis usadas no ajuste: apenas colunas climaticas com `dec` no nome.
- Variaveis excluidas do ajuste: rendimento, producao, area, valor de producao, ano, identificadores, municipio, latitude e longitude.
- Tratamento dos dados: sem agregacao, sem imputacao, sem padronizacao por desvio padrao e sem PCA no ajuste.
- Algoritmo de agrupamento: somente `KMeans`.
- Escolha de `k`: baseada em metricas internas e estabilidade, sem usar rendimento.
- Rendimento (`rendimento_kg_ha`): usado apenas depois do clustering, como interpretacao externa pos-hoc.

**Nota academica importante**

Este nao e um problema de classificacao supervisionada. Portanto, metricas como acuracia, precisao, recall, F1, matriz de confusao e ROC nao sao usadas como evidencia principal. Quartis de rendimento podem ser usados como referencia externa pos-hoc, mas nao sao ground truth do K-Means.


## 1. Configuração e bibliotecas

In [ ]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

# Garante compatibilidade cross-platform com loky/joblib
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap
from scipy.stats import kruskal
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    auc,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    roc_curve,
    silhouette_score,
)

try:
    from IPython.display import display
except ImportError:  # permite executar este notebook como script para validacao
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

# Paleta herdada do notebook unsupervised.ipynb.
# As quatro primeiras cores preservam a leitura baixo -> alto usada no original.
BASE_PALETTE = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]
EXTENDED_PALETTE = BASE_PALETTE + ["#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
PALETTE = EXTENDED_PALETTE
CMAP_DISC = ListedColormap(PALETTE)

DATA_CANDIDATES = [
    Path("data/processed/dataset_final.parquet"),
    Path("../../data/processed/dataset_final.parquet"),
    Path(r"C:/Users/bruno/Desktop/Pipeline_TCC/data/processed/dataset_final.parquet"),
]
PARQUET_PATH = next((p.resolve() for p in DATA_CANDIDATES if p.exists()), None)
if PARQUET_PATH is None:
    raise FileNotFoundError("dataset_final.parquet nao encontrado nos caminhos configurados.")

PROJECT_ROOT = PARQUET_PATH.parents[2]
OUTPUT_DIR = PROJECT_ROOT / "reports_kmeans_academico_dados_brutos"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARANA_BOUNDARY_PATH = PROJECT_ROOT / "data" / "processed" / "parana_boundary_geobr_2020.geojson"

ID_COL     = "cod_ibge"
NAME_COL   = "municipio"
YEAR_COL   = "ano"
TARGET_COL = "rendimento_kg_ha"
LAT_COL    = "latitude"
LON_COL    = "longitude"

K_VALUES           = range(2, 11)
RANDOM_STATE       = 42
N_INIT             = 10
MAX_ITER           = 500
STABILITY_SEEDS    = list(range(5))
BOOTSTRAPS         = 5
BOOTSTRAP_FRACTION = 0.80
MIN_CLUSTER_SHARE  = 0.05
SILHOUETTE_SAMPLE_SIZE = 1000
TOP_PROFILE_FEATURES = 12

print(f"Dataset: {PARQUET_PATH}")
print(f"Saida:   {OUTPUT_DIR}")


## 2. Funcoes auxiliares

As funcoes abaixo implementam o fluxo de forma reprodutivel: selecao de variaveis climaticas, montagem da matriz bruta, calculo de metricas internas e avaliacao de estabilidade. Nenhuma delas usa rendimento para ajustar ou escolher o K-Means.


In [ ]:
def identify_climate_features(df: pd.DataFrame) -> list[str]:
    """Seleciona somente variaveis climaticas decendiais, removendo coordenadas e colunas-alvo."""
    blocked = {ID_COL, NAME_COL, YEAR_COL, TARGET_COL, LAT_COL, LON_COL}
    return [
        c for c in df.columns
        if "dec" in c.lower()
        and "lat" not in c.lower()
        and "lon" not in c.lower()
        and c not in blocked
    ]


def build_raw_climate_dataset(df: pd.DataFrame, climate_cols: list[str]) -> pd.DataFrame:
    """Preserva as linhas brutas e apenas seleciona colunas climaticas e metadados pos-hoc."""
    required = [ID_COL, TARGET_COL, *climate_cols]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Colunas obrigatorias ausentes: {missing}")

    metadata_cols = [c for c in [ID_COL, NAME_COL, YEAR_COL, TARGET_COL, LAT_COL, LON_COL] if c in df.columns]
    selected_cols = metadata_cols + [c for c in climate_cols if c not in metadata_cols]
    raw = df.loc[:, selected_cols].copy()

    missing_values = int(raw[climate_cols].isna().sum().sum())
    if missing_values:
        raise ValueError(
            "As variaveis climaticas brutas contem valores ausentes. "
            "Esta versao nao imputa, nao remove linhas e nao altera os dados."
        )

    return raw.reset_index(drop=True)


def compute_internal_metrics(X: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    """Calcula metricas internas adequadas a clustering nao supervisionado."""
    counts = pd.Series(labels).value_counts().sort_index()
    silhouette_kwargs = {"random_state": RANDOM_STATE}
    if len(labels) > SILHOUETTE_SAMPLE_SIZE:
        silhouette_kwargs["sample_size"] = SILHOUETTE_SAMPLE_SIZE

    return {
        "silhouette":        float(silhouette_score(X, labels, **silhouette_kwargs)),
        "davies_bouldin":    float(davies_bouldin_score(X, labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
        "min_cluster_share": float(counts.min() / len(labels)),
    }


def fit_kmeans(X: np.ndarray, k: int, seed: int = RANDOM_STATE) -> tuple[np.ndarray, KMeans]:
    """Ajusta uma unica familia de modelo: K-Means."""
    model = KMeans(
        n_clusters=k,
        random_state=seed,
        n_init=N_INIT,
        max_iter=MAX_ITER,
        algorithm="lloyd",
    )
    labels = model.fit_predict(X)
    return labels, model


def mean_pairwise_ari(label_sets: list[np.ndarray]) -> tuple[float, float]:
    """Mede concordancia media e minima entre particoes usando ARI."""
    values = [adjusted_rand_score(a, b) for a, b in combinations(label_sets, 2)]
    if not values:
        return float("nan"), float("nan")
    return float(np.mean(values)), float(np.min(values))


def bootstrap_stability(
    X: np.ndarray,
    k: int,
    base_labels: np.ndarray,
    n_bootstraps: int = BOOTSTRAPS,
    fraction: float = BOOTSTRAP_FRACTION,
    seed: int = RANDOM_STATE,
) -> tuple[float, float]:
    """Compara a solucao completa com solucoes ajustadas em subamostras sem reposicao."""
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    sample_size = max(k + 1, int(round(n * fraction)))
    values = []

    for i in range(n_bootstraps):
        idx = np.sort(rng.choice(n, size=sample_size, replace=False))
        sample_labels, _ = fit_kmeans(X[idx], k, seed=seed + i + 1)
        values.append(adjusted_rand_score(base_labels[idx], sample_labels))

    return float(np.mean(values)), float(np.min(values))


def kruskal_effect_size(h_stat: float, n_obs: int, n_groups: int) -> float:
    """Epsilon squared para Kruskal-Wallis, limitado ao intervalo [0, 1]."""
    denom = max(n_obs - n_groups, 1)
    eps = (h_stat - n_groups + 1) / denom
    return float(np.clip(eps, 0.0, 1.0))


## 3. Leitura dos dados e selecao das variaveis

A selecao das variaveis e restritiva por desenho: somente colunas climaticas decendiais entram no espaco de agrupamento. Rendimento, ano, coordenadas e identificadores ficam preservados apenas para descricao e analise pos-hoc; eles nao entram no K-Means.


In [ ]:
df = pd.read_parquet(PARQUET_PATH)
climate_cols = identify_climate_features(df)
analysis_df = build_raw_climate_dataset(df, climate_cols)

info = pd.DataFrame({
    "indicador": [
        "linhas brutas na base original",
        "observacoes brutas usadas no K-Means",
        "features climaticas decendiais usadas no ajuste",
        "coluna externa de rendimento",
        "colunas preservadas fora do ajuste",
    ],
    "valor": [
        f"{len(df):,}",
        f"{len(analysis_df):,}",
        f"{len(climate_cols):,}",
        TARGET_COL,
        ", ".join([ID_COL, NAME_COL, YEAR_COL, TARGET_COL, LAT_COL, LON_COL]),
    ],
})

display(info)

raw_missing_summary = pd.DataFrame({
    "indicador": [
        "valores ausentes nas features climaticas",
        "linhas removidas antes do K-Means",
        "features criadas antes do K-Means",
    ],
    "valor": [
        int(analysis_df[climate_cols].isna().sum().sum()),
        0,
        0,
    ],
})
display(raw_missing_summary)

assert TARGET_COL not in climate_cols
assert LAT_COL    not in climate_cols
assert LON_COL    not in climate_cols
assert YEAR_COL   not in climate_cols


## 4. Matriz climatica bruta

A matriz do K-Means e formada diretamente por `df[climate_cols]`. Esta versao nao aplica imputacao, nao padroniza por desvio padrao, nao calcula componentes principais para o ajuste e nao agrega linhas.


In [ ]:
feature_matrix = analysis_df[climate_cols]

if feature_matrix.isna().any().any():
    raise ValueError(
        "A matriz climatica bruta contem valores ausentes. "
        "Esta versao nao faz imputacao nem remocao automatica de linhas."
    )

X_model = feature_matrix.to_numpy(dtype=np.float64, copy=True)

preprocess_summary = pd.DataFrame({
    "etapa": [
        "selecao das variaveis climaticas",
        "matriz usada no K-Means",
        "tratamentos nao aplicados",
    ],
    "descricao": [
        "somente colunas climaticas decendiais com dec no nome",
        f"{X_model.shape[0]} observacoes brutas x {X_model.shape[1]} features climaticas",
        "sem imputacao, sem padronizacao por desvio padrao, sem PCA e sem agregacao",
    ],
})

display(preprocess_summary)


## 5. Selecao de `k` com metricas internas

A selecao de `k` considera somente propriedades internas dos agrupamentos no espaco climatico bruto. A inercia e reportada para leitura do cotovelo, mas nao e suficiente sozinha porque tende a diminuir quando `k` aumenta.

Criterios usados:

- Silhouette: maior e melhor; calculada com amostra fixa quando a matriz bruta e grande.
- Davies-Bouldin: menor e melhor.
- Calinski-Harabasz: maior e melhor.
- Participacao minima do menor cluster: maior e melhor, usada para penalizar solucoes com grupos residuais.


In [ ]:
selection_rows = []
labels_by_k: dict[int, np.ndarray] = {}
models_by_k: dict[int, KMeans] = {}

for k in K_VALUES:
    labels, model = fit_kmeans(X_model, k, seed=RANDOM_STATE)
    labels_by_k[k] = labels
    models_by_k[k] = model
    metrics = compute_internal_metrics(X_model, labels)
    selection_rows.append({
        "k":      k,
        "inertia": float(model.inertia_),
        **metrics,
    })

selection = pd.DataFrame(selection_rows)
selection["rank_silhouette"]        = selection["silhouette"].rank(ascending=False, method="min")
selection["rank_davies_bouldin"]    = selection["davies_bouldin"].rank(ascending=True,  method="min")
selection["rank_calinski_harabasz"] = selection["calinski_harabasz"].rank(ascending=False, method="min")
selection["rank_balance"]           = selection["min_cluster_share"].rank(ascending=False, method="min")
selection["internal_rank_sum"] = selection[
    ["rank_silhouette", "rank_davies_bouldin", "rank_calinski_harabasz", "rank_balance"]
].sum(axis=1)

selection_pre_stability = selection.sort_values("internal_rank_sum").reset_index(drop=True)
display(selection_pre_stability)
selection_pre_stability.to_csv(OUTPUT_DIR / "kmeans_selection_pre_stability.csv", index=False, encoding="utf-8-sig")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8.2))
axes = axes.ravel()

plot_specs = [
    ("inertia",           "Inércia",          "menor com aumento de k", BASE_PALETTE[3]),
    ("silhouette",        "Silhouette",        "maior é melhor",         BASE_PALETTE[2]),
    ("davies_bouldin",    "Davies-Bouldin",    "menor é melhor",         BASE_PALETTE[0]),
    ("calinski_harabasz", "Calinski-Harabasz", "maior é melhor",         BASE_PALETTE[1]),
]

for ax, (metric, title, subtitle, color) in zip(axes, plot_specs):
    sns.lineplot(data=selection, x="k", y=metric, marker="o", ax=ax, color=color, linewidth=2.2)
    for _, row in selection.iterrows():
        ax.text(row["k"], row[metric], f"{row[metric]:.2f}", ha="center", va="bottom", fontsize=7)
    ax.set_title(f"{title} ({subtitle})")
    ax.set_xlabel("Número de clusters (k)")
    ax.set_ylabel(metric)
    ax.set_xticks(list(K_VALUES))
    ax.grid(axis="y", alpha=0.25)

plt.suptitle("Painel de seleção de k — métricas internas do K-Means", fontsize=12, y=1.01)
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(OUTPUT_DIR / "kmeans_internal_metrics.png", dpi=170, bbox_inches="tight")
plt.show()

## 6. Estabilidade das particoes

Alem das metricas internas, a escolha de `k` deve considerar estabilidade. Uma solucao academicamente mais defensavel nao deve depender excessivamente da semente inicial nem de pequenas mudancas na amostra.

- Estabilidade por sementes: ARI medio e ARI minimo entre particoes ajustadas com diferentes `random_state`.
- Estabilidade por bootstrap: ARI medio e ARI minimo entre a solucao completa e solucoes ajustadas em subamostras sem reposicao.

O ARI e adequado aqui porque compara particoes sem exigir que os rotulos dos clusters tenham o mesmo nome.


In [ ]:
stability_rows = []

for k in K_VALUES:
    seed_label_sets = [fit_kmeans(X_model, k, seed=seed)[0] for seed in STABILITY_SEEDS]
    seed_mean, seed_min = mean_pairwise_ari(seed_label_sets)
    boot_mean, boot_min = bootstrap_stability(
        X_model, k, labels_by_k[k],
        n_bootstraps=BOOTSTRAPS,
        fraction=BOOTSTRAP_FRACTION,
        seed=RANDOM_STATE,
    )
    stability_rows.append({
        "k":                            k,
        "seed_stability_ari_mean":      seed_mean,
        "seed_stability_ari_min":       seed_min,
        "bootstrap_stability_ari_mean": boot_mean,
        "bootstrap_stability_ari_min":  boot_min,
    })

stability = pd.DataFrame(stability_rows)
selection = selection.merge(stability, on="k", how="left")
selection["rank_seed_stability"]      = selection["seed_stability_ari_mean"].rank(ascending=False, method="min")
selection["rank_bootstrap_stability"] = selection["bootstrap_stability_ari_mean"].rank(ascending=False, method="min")
selection["consensus_rank_sum"] = selection[
    ["internal_rank_sum", "rank_seed_stability", "rank_bootstrap_stability"]
].sum(axis=1)

eligible = selection[selection["min_cluster_share"] >= MIN_CLUSTER_SHARE].copy()
if eligible.empty:
    eligible = selection.copy()

best_row = eligible.sort_values(
    ["consensus_rank_sum", "internal_rank_sum", "davies_bouldin"],
    ascending=[True, True, True],
).iloc[0]
BEST_K = int(best_row["k"])

selection_ranked = selection.sort_values("consensus_rank_sum").reset_index(drop=True)
display(selection_ranked)
selection_ranked.to_csv(OUTPUT_DIR / "kmeans_selection.csv", index=False, encoding="utf-8-sig")

print(f"k selecionado por criterios internos + estabilidade: {BEST_K}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.lineplot(
    data=selection, x="k", y="seed_stability_ari_mean",
    marker="o", ax=axes[0], label="Sementes",
    color=BASE_PALETTE[2], linewidth=2.2,
)
sns.lineplot(
    data=selection, x="k", y="bootstrap_stability_ari_mean",
    marker="o", ax=axes[0], label="Bootstrap",
    color=BASE_PALETTE[0], linewidth=2.2,
)
axes[0].set_title("Estabilidade das partições")
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("ARI médio")
axes[0].set_xticks(list(K_VALUES))
axes[0].set_ylim(0, 1.08)
axes[0].axhline(1.0, color="gray", linestyle=":", linewidth=0.8)
axes[0].legend(fontsize=9, loc="lower left")
axes[0].grid(axis="y", alpha=0.25)

bar_colors = [BASE_PALETTE[2] if int(k) == BEST_K else "#8fb9d9" for k in selection["k"]]
bars = axes[1].bar(selection["k"].astype(str), selection["consensus_rank_sum"], color=bar_colors, alpha=0.88)
for bar, value in zip(bars, selection["consensus_rank_sum"]):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
        f"{value:.0f}", ha="center", va="bottom", fontsize=8,
    )
axes[1].set_title("Ranking consensual: menor é melhor")
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Soma de ranks")
axes[1].grid(axis="y", alpha=0.25)

plt.suptitle("Estabilidade e escolha final do K-Means", fontsize=12, y=1.01)
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(OUTPUT_DIR / "kmeans_stability_and_rank.png", dpi=170, bbox_inches="tight")
plt.show()

## 6.1. Heatmap das metricas de selecao

Para tornar as escalas comparaveis, cada metrica do heatmap e normalizada para 0-100, sempre orientada para que valores maiores indiquem melhor posicao relativa.


In [ ]:
heat_data = selection.set_index("k")[[
    "silhouette",
    "davies_bouldin",
    "calinski_harabasz",
    "min_cluster_share",
    "seed_stability_ari_mean",
    "bootstrap_stability_ari_mean",
]].copy()

oriented = pd.DataFrame(index=heat_data.index)
oriented["Silhouette"]              = heat_data["silhouette"]
oriented["Davies-Bouldin invertido"]= -heat_data["davies_bouldin"]
oriented["Calinski-Harabasz"]       = heat_data["calinski_harabasz"]
oriented["Menor cluster"]           = heat_data["min_cluster_share"]
oriented["ARI sementes"]            = heat_data["seed_stability_ari_mean"]
oriented["ARI bootstrap"]           = heat_data["bootstrap_stability_ari_mean"]

heat_norm = oriented.copy()
for col in heat_norm.columns:
    col_min, col_max = heat_norm[col].min(), heat_norm[col].max()
    if np.isclose(col_min, col_max):
        heat_norm[col] = 100.0
    else:
        heat_norm[col] = 100 * (heat_norm[col] - col_min) / (col_max - col_min)

fig, ax = plt.subplots(figsize=(11.5, 5.2))
sns.heatmap(
    heat_norm,
    annot=True, fmt=".1f", cmap="RdYlGn",
    vmin=0, vmax=100, linewidths=0.5, linecolor="white",
    ax=ax, annot_kws={"fontsize": 9},
)
ax.set_title(
    "Heatmap de seleção de k — métricas internas e estabilidade\n"
    "Verde = melhor posição relativa; vermelho = pior"
)
ax.set_ylabel("k")
ax.set_xlabel("")
ax.tick_params(axis="x", labelrotation=15)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_selection_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()

## 6.2. Radar dos critérios não supervisionados

O radar segue o padrão visual do notebook original, mas substitui métricas supervisionadas por critérios de clustering. Os valores estão normalizados entre 0 e 1, sempre orientados para que valores maiores indiquem melhor desempenho relativo.

In [ ]:
radar_metrics = [
    "Silhouette",
    "Davies-Bouldin invertido",
    "Calinski-Harabasz",
    "Menor cluster",
    "ARI sementes",
    "ARI bootstrap",
]
radar_k_values = selection_ranked["k"].head(min(4, len(selection_ranked))).astype(int).tolist()

angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8.5, 8), subplot_kw=dict(polar=True))

for idx, k_value in enumerate(radar_k_values):
    values = (heat_norm.loc[k_value, radar_metrics] / 100).to_numpy().tolist()
    values += values[:1]
    color = PALETTE[idx % len(PALETTE)]
    ax.plot(angles, values, color=color, lw=2.2, label=f"k={k_value}")
    ax.fill(angles, values, color=color, alpha=0.16)

ax.plot(angles, [1] * len(angles), "k--", lw=0.7, alpha=0.35)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7)
ax.set_title("Radar de critérios internos e estabilidade — K-Means", fontsize=12, pad=18)
ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.12), fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_radar_internal_stability.png", dpi=170, bbox_inches="tight")
plt.show()

## 7. Modelo final K-Means

Apos a selecao de `k`, o modelo final e ajustado uma unica vez com a semente definida no protocolo. A coluna `cluster_kmeans` preserva o rotulo original do algoritmo. A coluna `cluster_apresentacao` apenas reordena os grupos para facilitar leitura dos resultados pos-hoc.


In [ ]:
final_labels, final_model = fit_kmeans(X_model, BEST_K, seed=RANDOM_STATE)
analysis_df = analysis_df.copy()
analysis_df["cluster_kmeans"] = final_labels

# Ordenacao apenas expositiva: do menor para o maior rendimento mediano pos-hoc.
cluster_order = (
    analysis_df.groupby("cluster_kmeans")[TARGET_COL]
    .median().sort_values().index.tolist()
)
presentation_map = {cluster: i + 1 for i, cluster in enumerate(cluster_order)}
analysis_df["cluster_apresentacao"] = analysis_df["cluster_kmeans"].map(presentation_map)

CLUSTER_LABELS = [f"Cluster {i}" for i in range(1, BEST_K + 1)]
cluster_palette = PALETTE[:BEST_K]
CMAP_CLUSTERS   = ListedColormap(cluster_palette)

final_metrics = {
    "k":      BEST_K,
    "inertia": float(final_model.inertia_),
    **compute_internal_metrics(X_model, final_labels),
}
display(pd.DataFrame([final_metrics]))

label_cols = [ID_COL]
for optional_col in [NAME_COL, YEAR_COL, LAT_COL, LON_COL]:
    if optional_col in analysis_df.columns and optional_col not in label_cols:
        label_cols.append(optional_col)
for result_col in [TARGET_COL, "cluster_kmeans", "cluster_apresentacao"]:
    if result_col in analysis_df.columns and result_col not in label_cols:
        label_cols.append(result_col)

analysis_df[label_cols].to_csv(OUTPUT_DIR / "kmeans_final_labels.csv", index=False, encoding="utf-8-sig")


## 8. Perfil dos clusters e validação externa pós-hoc

A tabela abaixo descreve os grupos usando rendimento apenas após o ajuste. O teste de Kruskal-Wallis avalia se a distribuição de rendimento difere entre clusters, mas não transforma o estudo em supervisionado e não estabelece causalidade.

In [ ]:
cluster_summary = (
    analysis_df.groupby("cluster_apresentacao")
    .agg(
        n_observacoes=(ID_COL, "count"),
        rendimento_media=(TARGET_COL, "mean"),
        rendimento_mediana=(TARGET_COL, "median"),
        rendimento_q25=(TARGET_COL, lambda s: s.quantile(0.25)),
        rendimento_q75=(TARGET_COL, lambda s: s.quantile(0.75)),
    )
    .reset_index()
)
cluster_summary["participacao_pct"] = 100 * cluster_summary["n_observacoes"] / len(analysis_df)
cluster_summary["rendimento_iqr"]   = cluster_summary["rendimento_q75"] - cluster_summary["rendimento_q25"]
cluster_summary = cluster_summary[[
    "cluster_apresentacao", "n_observacoes", "participacao_pct",
    "rendimento_media", "rendimento_mediana", "rendimento_iqr",
]]

display(cluster_summary)
cluster_summary.to_csv(OUTPUT_DIR / "kmeans_cluster_summary.csv", index=False, encoding="utf-8-sig")

reference_quartile = pd.qcut(analysis_df[TARGET_COL], q=4, labels=False, duplicates="drop").astype(int)
yield_groups = [
    analysis_df.loc[analysis_df["cluster_kmeans"] == c, TARGET_COL].to_numpy()
    for c in sorted(analysis_df["cluster_kmeans"].unique())
]

h_stat, p_value = kruskal(*yield_groups)
epsilon_sq = kruskal_effect_size(h_stat, n_obs=len(analysis_df), n_groups=BEST_K)

external_validation = pd.DataFrame([{
    "ari_vs_quartis_rendimento": adjusted_rand_score(reference_quartile, final_labels),
    "nmi_vs_quartis_rendimento": normalized_mutual_info_score(reference_quartile, final_labels),
    "ami_vs_quartis_rendimento": adjusted_mutual_info_score(reference_quartile, final_labels),
    "kruskal_h_rendimento":      float(h_stat),
    "kruskal_p_rendimento":      float(p_value),
    "kruskal_epsilon_sq":        epsilon_sq,
}])

display(external_validation)
external_validation.to_csv(OUTPUT_DIR / "kmeans_external_validation_posthoc.csv", index=False, encoding="utf-8-sig")

contingency = pd.crosstab(
    reference_quartile + 1,
    analysis_df["cluster_apresentacao"],
    rownames=["quartil_rendimento"],
    colnames=["cluster_apresentacao"],
    normalize="index",
) * 100

display(contingency.round(2))


## 8.1. Apêndice exploratório: curva ROC pós-hoc

Esta curva ROC não é usada para validar nem escolher o K-Means. Ela é mantida apenas como visualização exploratória de associação entre a proximidade ao cluster de maior rendimento mediano e uma referência binária externa de rendimento alto (`Q3` e `Q4`) contra rendimento baixo (`Q1` e `Q2`).

A pontuação usada na curva é derivada da distância ao centroide do cluster pós-hoc de maior rendimento mediano. Portanto, ela não representa probabilidade calibrada nem transforma o estudo em classificação supervisionada.

In [ ]:
if BEST_K < 2:
    print("ROC pos-hoc nao gerada: e necessario pelo menos dois clusters.")
else:
    # Q3-Q4 = rendimento alto; Q1-Q2 = rendimento baixo
    y_binary = (reference_quartile >= 2).astype(int)

    high_cluster_presentation = int(
        cluster_summary.sort_values("rendimento_mediana").iloc[-1]["cluster_apresentacao"]
    )
    inverse_presentation_map = {v: k for k, v in presentation_map.items()}
    high_cluster_raw = inverse_presentation_map[high_cluster_presentation]

    distances = final_model.transform(X_model)
    inverse_distance = 1.0 / (distances + 1e-9)
    posthoc_score = inverse_distance[:, high_cluster_raw] / inverse_distance.sum(axis=1)

    fpr, tpr, thresholds = roc_curve(y_binary, posthoc_score)
    auc_value = auc(fpr, tpr)

    pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": thresholds}).to_csv(
        OUTPUT_DIR / "kmeans_roc_posthoc_binary.csv", index=False, encoding="utf-8-sig"
    )

    fig, ax = plt.subplots(figsize=(7.2, 6.4))
    ax.plot(fpr, tpr, color=BASE_PALETTE[3], lw=2.4, label=f"AUC pos-hoc = {auc_value:.4f}")
    ax.fill_between(fpr, tpr, color=BASE_PALETTE[3], alpha=0.13)
    ax.plot([0, 1], [0, 1], "k--", lw=0.9, alpha=0.5, label="Referencia aleatoria")

    youden_idx = int(np.argmax(tpr - fpr))
    ax.scatter(
        fpr[youden_idx], tpr[youden_idx],
        s=70, color=BASE_PALETTE[0], edgecolors="black", zorder=5,
        label=f"Youden ({fpr[youden_idx]:.2f}, {tpr[youden_idx]:.2f})",
    )

    ax.set_title("Curva ROC pos-hoc - associacao com rendimento alto")
    ax.set_xlabel("Taxa de falsos positivos")
    ax.set_ylabel("Taxa de verdadeiros positivos")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9, loc="lower right")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "kmeans_roc_posthoc_binary.png", dpi=170, bbox_inches="tight")
    plt.show()


## 9. Interpretacao climatica dos grupos

Como o K-Means foi ajustado diretamente nas variaveis climaticas brutas, a interpretacao climatica e feita pos-hoc pela diferenca media bruta das variaveis em cada cluster. Valores positivos indicam que o cluster esta acima da media geral naquela variavel; valores negativos indicam abaixo da media geral.


In [ ]:
global_mean_raw = X_model.mean(axis=0)
profile_rows = []

for cluster in sorted(analysis_df["cluster_kmeans"].unique()):
    idx = analysis_df["cluster_kmeans"].to_numpy() == cluster
    cluster_mean = X_model[idx].mean(axis=0)
    diff = cluster_mean - global_mean_raw
    top_idx = np.argsort(np.abs(diff))[::-1][:TOP_PROFILE_FEATURES]

    for rank, feature_idx in enumerate(top_idx, 1):
        profile_rows.append({
            "cluster_kmeans":              int(cluster),
            "cluster_apresentacao":        int(presentation_map[cluster]),
            "rank_abs_diff":               rank,
            "feature":                     climate_cols[feature_idx],
            "media_cluster_bruta":         float(cluster_mean[feature_idx]),
            "diferenca_vs_media_global":   float(diff[feature_idx]),
            "abs_diferenca":               float(abs(diff[feature_idx])),
        })

feature_profile = pd.DataFrame(profile_rows).sort_values(["cluster_apresentacao", "rank_abs_diff"])
display(feature_profile)
feature_profile.to_csv(OUTPUT_DIR / "kmeans_feature_profile.csv", index=False, encoding="utf-8-sig")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Boxplot: quartis de rendimento como referencia externa
quartile_palette = BASE_PALETTE[:4]
quartile_labels  = ["Q1 Baixo", "Q2 Medio-Baixo", "Q3 Medio-Alto", "Q4 Alto"]
reference_plot = analysis_df.copy()
reference_plot["quartil_rendimento"] = reference_quartile + 1

data_reference = [
    reference_plot.loc[reference_plot["quartil_rendimento"] == i, TARGET_COL].values
    for i in range(1, 5)
]
bp1 = axes[0].boxplot(
    data_reference,
    labels=quartile_labels,
    patch_artist=True,
    showfliers=True,
    flierprops=dict(marker="o", markersize=3, alpha=0.4),
)
for patch, color in zip(bp1["boxes"], quartile_palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_ylabel("Rendimento medio (kg/ha)")
axes[0].set_title("Quartis de rendimento (referencia pos-hoc)")
axes[0].tick_params(axis="x", labelrotation=15)
axes[0].grid(axis="y", alpha=0.25)

# Boxplot: clusters K-Means
data_clusters = [
    analysis_df.loc[analysis_df["cluster_apresentacao"] == i, TARGET_COL].values
    for i in range(1, BEST_K + 1)
]
bp2 = axes[1].boxplot(
    data_clusters,
    labels=CLUSTER_LABELS,
    patch_artist=True,
    showfliers=True,
    flierprops=dict(marker="o", markersize=3, alpha=0.4),
)
for patch, color in zip(bp2["boxes"], cluster_palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_ylabel("Rendimento medio (kg/ha)")
axes[1].set_title(f"Clusters K-Means (k={BEST_K}) - analise pos-hoc")
axes[1].tick_params(axis="x", labelrotation=15)
axes[1].grid(axis="y", alpha=0.25)

plt.suptitle("Distribuicao de rendimento: referencia externa vs. K-Means", fontsize=12, y=1.01)
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(OUTPUT_DIR / "kmeans_yield_boxplots_original_style.png", dpi=170, bbox_inches="tight")
plt.show()


## 9.1. Visualizacao dos clusters em variaveis climaticas brutas

Esta figura mostra os clusters em duas variaveis climaticas brutas selecionadas pelo maior contraste medio pos-hoc. Ela e descritiva: a selecao formal do modelo permanece baseada nas metricas internas e na estabilidade.


In [ ]:
plot_features = feature_profile["feature"].drop_duplicates().head(2).tolist()
if len(plot_features) < 2:
    plot_features = climate_cols[:2]

plot_df = analysis_df[["cluster_apresentacao", TARGET_COL, *plot_features]].copy()
plot_df["cluster_apresentacao"] = plot_df["cluster_apresentacao"].astype(str)

fig, ax = plt.subplots(figsize=(8.8, 6.4))
sns.scatterplot(
    data=plot_df, x=plot_features[0], y=plot_features[1],
    hue="cluster_apresentacao", palette=cluster_palette,
    s=46, edgecolor="black", linewidth=0.25, alpha=0.82, ax=ax,
)
ax.set_title("K-Means em duas variaveis climaticas brutas")
ax.set_xlabel(plot_features[0])
ax.set_ylabel(plot_features[1])
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_raw_feature_scatter.png", dpi=170, bbox_inches="tight")
plt.show()


In [ ]:
if LAT_COL in analysis_df.columns and LON_COL in analysis_df.columns:
    fig, ax = plt.subplots(figsize=(8, 8))
    sns.scatterplot(
        data=analysis_df, x=LON_COL, y=LAT_COL,
        hue=analysis_df["cluster_apresentacao"].astype(str),
        palette=cluster_palette,
        s=40, edgecolor="black", linewidth=0.25, alpha=0.72, ax=ax,
    )
    ax.set_title("Clusters K-Means nas observacoes brutas")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "kmeans_spatial_scatter.png", dpi=170, bbox_inches="tight")
    plt.show()
else:
    print("Colunas de latitude/longitude ausentes; mapa de dispersao nao foi gerado.")


## 9.2. Interpolacao espacial RBF

A interpolacao RBF foi desativada nesta copia porque criaria uma superficie derivada a partir dos pontos e exigiria tratamento espacial adicional. Para manter o pedido de dados brutos, esta versao preserva apenas o mapa de dispersao das observacoes originais.


In [ ]:
print(
    "Interpolacao RBF nao executada nesta versao: "
    "a copia de dados brutos nao agrega, nao interpola e nao cria superficies derivadas."
)


## 10. Síntese acadêmica gerada pelo notebook

A célula final registra uma nota metodológica curta com os principais resultados e limitações. Ela também salva a nota em Markdown para uso em relatório.

In [ ]:
best     = selection.loc[selection["k"] == BEST_K].iloc[0]
external = external_validation.iloc[0]

method_note = f"""# Nota metodologica - K-Means nao supervisionado com dados brutos

Objetivo:
- Agrupar observacoes brutas por padroes climaticos decendiais, sem usar rendimento, producao, area, coordenadas ou identificadores no ajuste.

Algoritmo:
- K-Means foi o unico algoritmo de clustering utilizado.
- A variacao de k entre {min(K_VALUES)} e {max(K_VALUES)} foi tratada como selecao de hiperparametro, nao como comparacao de algoritmos.

Matriz de entrada:
- Unidade de analise: linha bruta da base original.
- Variaveis de ajuste: {len(climate_cols)} colunas climaticas decendiais.
- Sem agregacao por municipio.
- Sem imputacao.
- Sem padronizacao por desvio padrao.
- Sem PCA no ajuste.

Escolha de k:
- k selecionado: {BEST_K}.
- Silhouette: {best['silhouette']:.4f} (amostra fixa quando aplicavel).
- Davies-Bouldin: {best['davies_bouldin']:.4f}.
- Calinski-Harabasz: {best['calinski_harabasz']:.2f}.
- Menor participacao de cluster: {best['min_cluster_share']:.2%}.
- Estabilidade por sementes, ARI medio: {best['seed_stability_ari_mean']:.4f}.
- Estabilidade por sementes, ARI minimo: {best['seed_stability_ari_min']:.4f}.
- Estabilidade por bootstrap, ARI medio: {best['bootstrap_stability_ari_mean']:.4f}.
- Estabilidade por bootstrap, ARI minimo: {best['bootstrap_stability_ari_min']:.4f}.

Validacao externa pos-hoc:
- Rendimento foi usado apenas depois do clustering.
- ARI contra quartis de rendimento: {external['ari_vs_quartis_rendimento']:.4f}.
- NMI contra quartis de rendimento: {external['nmi_vs_quartis_rendimento']:.4f}.
- Kruskal-Wallis p-valor: {external['kruskal_p_rendimento']:.6g}.
- Epsilon squared: {external['kruskal_epsilon_sq']:.4f}.

Interpretacao:
- Os clusters descrevem estrutura climatica nas observacoes brutas.
- Associacao com rendimento deve ser interpretada como evidencia exploratoria, nao causal.
- Fatores nao climaticos, como solo, manejo, cultivar, tecnologia, pragas e mercado, podem influenciar o rendimento.
"""

print(method_note)
(OUTPUT_DIR / "nota_metodologica_kmeans.md").write_text(method_note, encoding="utf-8")
